In [12]:
import os
import tarfile

import numpy as np
import xarray as xr
import zarr

In [2]:
print(zarr.__version__)
print(xr.__version__)
print(np.__version__)

3.1.5.dev2734+g596fd7f91
2025.12.0
2.4.0


# Testing zarr stores with sample data

In [18]:
times = np.arange('2023-01-01', '2023-01-03', dtype='datetime64[D]')
lats = [34, 35, 36]
lons = [-118, -117, -116]
variableDict= {
    'temperature' : {
            'values' : np.random.rand(2, 3, 3) * 30 + 273.15, # (time, lat, lon)
            'units' : "K"
                    },
    'precipitation' : {
            'values' : np.random.rand(2, 3, 3),  # (time, lat, lon)
            'units' : "mm/day"
                    }
}
zarrStorePrefix='testStore'
zarrStoreList = []

In [5]:
def createZarrStore( varName, index ):
    # Create the Dataset
    ds = xr.Dataset(
        data_vars={
            f"{varName}": (
                            ("time", "lat", "lon"),
                            variableDict[varName]['values'],
                            {"units": variableDict[varName]['units']}),
        },
        coords={
            "time": times,
            "lat": lats,
            "lon": lons
        },
        attrs={"description": "Sample weather data"}
    )
    #Convert and save to zarr store with 'index' in the name.
    zarrStoreName = f'{zarrStorePrefix}{index}Dev.zarr'
    ds.to_zarr( f'{zarrStoreName}', mode='w', zarr_format=3, consolidated = True )
    zarrStoreList.append(zarrStoreName)

In [6]:
count = 1
for variable in variableDict.keys():
    createZarrStore( variable, count )
    count += 1

/work/bk1414/k204247/staczarrDev/tools/myRepo/zarr-python/src/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
/work/bk1414/k204247/staczarrDev/tools/myRepo/zarr-python/src/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


In [14]:
def scan_dir( dirName ):
    with os.scandir( dirName ) as it:
        for entry in it:
            if entry.is_file():
                list_of_vars.append( entry.path )
            elif entry.is_dir():
                scan_dir( entry.path )

In [15]:
def createTarStoreFromZarrStore(zarrStoreName):
    zarrBaseName=zarrStoreName.split('.')[0]
    tarFileName=f'{zarrBaseName}.tar'

    scan_dir( zarrStoreName )
    print(list_of_vars)

    try:
        with tarfile.open(tarFileName, "w") as tar:
            for name in list_of_vars:
                print(f"Adding file {name} to {tarFileName}\n")
                tar.add( name, arcname=name.replace( zarrStoreName + os.path.sep, '' ) )
        tar.close()
    except Exception as e:
        print(f"Exception occured: {e}")

In [16]:
for zarrStore in zarrStoreList:
    list_of_vars=[]
    createTarStoreFromZarrStore( zarrStore )

['testStore1Dev.zarr/time/zarr.json', 'testStore1Dev.zarr/time/c/0', 'testStore1Dev.zarr/temperature/zarr.json', 'testStore1Dev.zarr/temperature/c/0/0/0', 'testStore1Dev.zarr/zarr.json', 'testStore1Dev.zarr/lon/zarr.json', 'testStore1Dev.zarr/lon/c/0', 'testStore1Dev.zarr/lat/zarr.json', 'testStore1Dev.zarr/lat/c/0']
Adding file testStore1Dev.zarr/time/zarr.json to testStore1Dev.tar

Adding file testStore1Dev.zarr/time/c/0 to testStore1Dev.tar

Adding file testStore1Dev.zarr/temperature/zarr.json to testStore1Dev.tar

Adding file testStore1Dev.zarr/temperature/c/0/0/0 to testStore1Dev.tar

Adding file testStore1Dev.zarr/zarr.json to testStore1Dev.tar

Adding file testStore1Dev.zarr/lon/zarr.json to testStore1Dev.tar

Adding file testStore1Dev.zarr/lon/c/0 to testStore1Dev.tar

Adding file testStore1Dev.zarr/lat/zarr.json to testStore1Dev.tar

Adding file testStore1Dev.zarr/lat/c/0 to testStore1Dev.tar

['testStore2Dev.zarr/precipitation/zarr.json', 'testStore2Dev.zarr/precipitation/c/0